# Magnetosheath and magnetosphere in PGSM

`frame="pgsm"` pools measurements by symmetry (Michotte de Welle 2024): the magnetosheath is
selected by IMF cone angle and rotated to one target IMF clock angle; the magnetosphere uses the
dipole-tilt symmetry. See the PGSM section of the user guide.

```{note}
These pages run on a small real-data sample served locally (dataset version
`2026.0-docs-sample`). On the public server, run the `mango.count(...)` cell first and narrow
the spacecraft and time window if the numbers are large.
```

In [ ]:
import matplotlib.pyplot as plt

import space_mango as mango

WINDOW = dict(spacecraft="THA", start="2008-08-01", stop="2008-09-01")

In [ ]:
mango.count("magnetosheath", frame="pgsm", cone=[45, 135], **WINDOW)

In [ ]:
r = mango.get_data("magnetosheath", frame="pgsm", cone=[45, 135], clock=180,
                   columns=["Time", "Np"], **WINDOW)
r.to_polars().head()

The same rows rotated to two clock angles: only the local transform runs again.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, clock in zip(axes, [180, 90], strict=True):
    d = mango.get_data("magnetosheath", frame="pgsm", cone=[45, 135], clock=clock,
                       columns=["Time"], **WINDOW).to_polars()
    sc = ax.scatter(d["Y_pgsm_norm"], d["Z_pgsm_norm"], c=d["Bz_pgsm"], s=4, cmap="RdBu_r")
    ax.set(title=f"clock = {clock}°", xlabel="Y_pgsm_norm (R_E)", aspect="equal")
axes[0].set_ylabel("Z_pgsm_norm (R_E)")
fig.colorbar(sc, ax=axes, label="Bz_pgsm (nT)");

## Magnetosphere: tilt symmetry

Rows with tilt in the range are kept as measured; rows with the opposite tilt are added as
mirror images (`mirrored`).

In [ ]:
m = mango.get_data("magnetosphere", frame="pgsm", tilt=[10, 35], columns=["Time"], **WINDOW)
m.to_polars().group_by("mirrored").len()